# Advanced File Handling: The `with` Statement

The `with` statement is the modern, standard, and recommended way to handle files in Python. It acts as a **context manager**.

### Why use `with`? Two big reasons:

1.  **Automatic Resource Management**: The `with` statement guarantees that the file's `close()` method is called automatically as soon as you exit the `with` block. You **never** have to write `file.close()` again. This prevents resource leaks.
2.  **Exception Safety**: What if an error occurs while you're processing the file? In a manual `open()`/`close()` setup, your `close()` call might be skipped, leaving the file open. The `with` statement ensures the file is closed **even if exceptions happen** inside the block.

Let's compare the old way with the `with` statement.

In [ ]:
# First, let's create a dummy file to work with.
with open("poem.txt", "w") as f:
    f.write("In realms of code, where logic takes its flight,\n")
    f.write("A silent cursor blinks in pale moonlight.\n")

# --- Method 1: Manual open() and close() (The Old Way) ---
print("--- Manual Method ---")
try:
    file_handle = open("poem.txt", 'r')
    content = file_handle.read()
    print("File content read successfully.")
    print(f"Is file closed? {file_handle.closed}")
finally:
    # You MUST remember to close the file in a finally block
    # to ensure it closes even if errors occur.
    file_handle.close()
    print("File closed in 'finally' block.")
    print(f"Is file closed? {file_handle.closed}")

print("\n" + "-"*20 + "\n")

# --- Method 2: Using the `with` statement (The Pythonic Way) ---
print("--- `with` Statement Method ---")
with open("poem.txt", 'r') as f:
    content = f.read()
    print("File content read successfully.")
    # The file is open inside this block
    print(f"Inside 'with' block, is file closed? {f.closed}")

# Once the block is exited, the file is automatically closed.
print("Exited 'with' block.")
print(f"Outside 'with' block, is file closed? {f.closed}")

### Common Mistake & Key Takeaway

**Common Mistake**: Trying to use the file object *after* the `with` block has finished. The file is closed, and any attempt to read or write will result in a `ValueError`.

```python
with open('poem.txt', 'r') as f:
    # This is fine
    print("Reading inside the block...")

# This will raise a ValueError: I/O operation on closed file.
# f.read() 
```

**Key Takeaway**: Always use the `with` statement for file I/O. It's cleaner, safer, and less error-prone. Perform all file operations **inside** the indented `with` block.

# Advanced File Modes: `r+`, `w+`, `a+`, `rb`, `wb`

Beyond the basic `'r'`, `'w'`, and `'a'` modes, Python offers more powerful modes that combine reading, writing, and other behaviors. The `+` sign typically adds a capability (usually writing to a read mode or reading to a write mode).

### Comparison Table

| Mode | Description | Reads? | Writes? | Creates File? | Truncates File? | Pointer Position | Use Case |
| :--- | :--- | :---: | :---: | :---: | :---: | :--- | :--- |
| `r+` | Read and Write | ✅ | ✅ | ❌ | ❌ | Beginning | Read a file and overwrite parts of it in-place. |
| `w+` | Write and Read | ✅ | ✅ | ✅ | ✅ | Beginning | Start with a blank slate, write new content, then read it back. | 
| `a+` | Append and Read | ✅ | ✅ | ✅ | ❌ | End | Add content to the end of a file and then read the whole file. |
| `rb` | Read Binary | ✅ | ❌ | ❌ | ❌ | Beginning | Read non-text files like images, audio, or model weights. |
| `wb` | Write Binary | ❌ | ✅ | ✅ | ✅ | Beginning | Write or create non-text files. |

**Truncates File?**: This means the file's existing content is deleted the moment it's opened.

In [ ]:
# Let's create a file to demonstrate the modes
with open("myfile.txt", "w") as f:
    f.write("Line 1\nLine 2\nLine 3")

# --- Demo: 'r+' (Read and Write) ---
print("--- 'r+' Demo ---")
with open("myfile.txt", "r+") as f:
    content = f.read() # Reads the whole file, pointer is now at the end
    print(f"Read content:\n{content}")
    f.write("\nLine 4") # Writes 'Line 4' at the end
    print("'Line 4' was written.")

# --- Demo: 'w+' (Write and Read) --- 
# DANGER: This mode first truncates (deletes) the file content!
print("\n--- 'w+' Demo ---")
with open("myfile.txt", "w+") as f:
    print("File opened with 'w+'. Content is now gone.")
    f.write("New Content")
    # Pointer is at the end. To read, we must move it back to the start.
    f.seek(0)
    new_content = f.read()
    print(f"Read back content: '{new_content}'")

# --- Demo: 'a+' (Append and Read) ---
print("\n--- 'a+' Demo ---")
with open("myfile.txt", "a+") as f:
    # Pointer starts at the end for writing
    f.write(" Appended")
    # To read the whole file, we must seek back to the start
    f.seek(0)
    full_content = f.read()
    print(f"Full content after append: '{full_content}'")

# --- Demo: 'wb' (Write Binary) ---
# We'll write a sequence of bytes, not a string.
with open("data.bin", "wb") as f:
    byte_data = b'\x01\x02\x03\xDE\xAD\xBE\xEF'
    f.write(byte_data)

# --- Demo: 'rb' (Read Binary) ---
with open("data.bin", "rb") as f:
    read_bytes = f.read()
    print(f"\nRead binary data: {read_bytes}")

### Common Mistake & Key Takeaway

**Common Mistake**: Confusing `r+` and `w+`. A student might use `w+` intending to read a file and then modify it, but `w+` **deletes all content** upon opening. `r+` is for reading first, then modifying.

**Key Takeaway**: The `+` modes are powerful but require careful management of the file pointer (`seek()`). `r+` modifies, `w+` replaces, and `a+` adds. Use binary modes (`rb`, `wb`) for anything that isn't plain text.

# Reading Methods: `read()`, `readline()`, `readlines()`, and Iteration

How you read a file can have a massive impact on your program's memory usage. This is critically important in data science and machine learning, where you might be processing datasets that are many gigabytes in size.

| Method | How it Works | Memory Usage | When to Use |
| :--- | :--- | :--- | :--- |
| `file.read()` | Reads the **entire** file content into a single string. | **High** | Small files where you need the whole content at once. | 
| `file.readline()` | Reads just **one** line from the file, including the `\n`. | **Low** | Processing a file line-by-line where you only need one line in memory at a time. | 
| `file.readlines()` | Reads the **entire** file into a list of strings, where each string is a line. | **High** | Small files where you want to work with a list of all lines. | 
| `for line in file:` | Iterates over the file object line-by-line. | **Low** | The most common and Pythonic way to process a large file line-by-line. It's memory-efficient and readable. |

In [ ]:
# Let's create a slightly larger text file
with open("text.txt", "w") as f:
    f.write("First line.\n")
    f.write("Second line.\n")
    f.write("Third line.\n")

# --- read() --- 
print("--- read() ---")
with open("text.txt", 'r') as f:
    content = f.read()
    print(repr(content)) # repr() shows hidden characters like \n

# --- readline() --- 
print("--- readline() ---")
with open("text.txt", 'r') as f:
    line1 = f.readline()
    line2 = f.readline()
    print(f"Line 1: {repr(line1)}")
    print(f"Line 2: {repr(line2)}")

# --- readlines() --- 
print("--- readlines() ---")
with open("text.txt", 'r') as f:
    all_lines = f.readlines()
    print(all_lines)

# --- Iteration (The Best Way for Large Files) --- 
print("--- Iteration ---")
with open("text.txt", 'r') as f:
    for line in f:
        # .strip() is often used to remove leading/trailing whitespace, including the newline character
        print(f"Processing line: {line.strip()}")

### Common Mistake & Key Takeaway

**Common Mistake**: Using `read()` or `readlines()` on a very large file (e.g., a 10GB log file or dataset). This will attempt to load the entire file into RAM, which can crash your program or computer.

**Key Takeaway**: For large files, **always** prefer iterating directly over the file object (`for line in file:`). It is the most memory-efficient and scalable approach. It gives you the best of both worlds: low memory usage and clean, readable code.

# Writing Methods: `write()` vs `writelines()`

Just as there are multiple ways to read, there are a couple of ways to write.

| Method | What it Accepts | How it Works |
| :--- | :--- | :--- |
| `file.write(string)` | A single string. | Writes the string to the file. **It does not add a newline character automatically.** |
| `file.writelines(list_of_strings)` | A list (or any iterable) of strings. | Iterates over the list and writes each string to the file. **It does not add newlines between strings.** |

In [ ]:
# --- write() --- 
print("--- write() ---")
with open("output1.txt", "w") as f:
    f.write("Hello")
    f.write("World") # This will be written immediately after "Hello"
    f.write("\n") # You must add newlines manually
    f.write("This is a new line.")

with open("output1.txt", 'r') as f:
    print(f.read())

# --- writelines() --- 
print("\n--- writelines() ---")
lines_to_write = ["Alpha\n", "Bravo\n", "Charlie\n"]
with open("output2.txt", "w") as f:
    # Note that the newline characters are already in the list items
    f.writelines(lines_to_write)

with open("output2.txt", 'r') as f:
    print(f.read())

### Common Mistake & Key Takeaway

**Common Mistake**: Assuming `writelines()` adds newline characters between the list items. It does not. It simply concatenates the strings from the list and writes them. You must include `\n` in your strings if you want them on separate lines.

**Key Takeaway**: `write()` is for writing individual strings. `writelines()` is a convenience method for writing all items from an iterable. In both cases, you are in complete control of formatting, including newlines.

# File Pointer (Cursor): `tell()` and `seek()`

Imagine a cursor in a text editor. When you read or write to a file, Python uses a similar internal pointer to keep track of your current position. 

- **Analogy**: Think of it as a bookmark in a book. When you `read()`, you move the bookmark to the end of what you just read.

This concept is crucial for understanding modes like `r+` and `w+`, where you might need to read something and then move the pointer to a specific location to write.

| Method | What it Does |
| :--- | :--- |
| `file.tell()` | Returns an integer giving the file pointer's current position in bytes from the beginning of the file. |
| `file.seek(offset, from_what)` | Changes the position of the file pointer. |

**`seek()` Parameters:**
- `offset`: The number of bytes to move.
- `from_what` (optional): Defines the starting point for the move.
  - `0`: (Default) From the beginning of the file.
  - `1`: From the current position.
  - `2`: From the end of the file.

In [ ]:
with open("seek_demo.txt", "w") as f:
    f.write("0123456789abcdef")

with open("seek_demo.txt", "r+") as f:
    print(f"Opening file. Pointer at: {f.tell()}") # Starts at 0
    
    # Read the first 5 bytes
    first_five = f.read(5)
    print(f"Read 5 bytes: '{first_five}'")
    print(f"Pointer is now at: {f.tell()}") # At position 5
    
    # Let's go back to the beginning and overwrite something
    f.seek(0) # Move pointer to byte 0
    print(f"Seek(0). Pointer is now at: {f.tell()}")
    
    f.write("HELLO") # Overwrite the first 5 characters
    print(f"Wrote 'HELLO'. Pointer is now at: {f.tell()}")
    
    # Go to the end of the file and add something
    f.seek(0, 2) # 0 bytes from the end of the file (i.e., AT the end)
    print(f"Seek(0, 2). Pointer is now at: {f.tell()}")
    f.write("_end")

# Let's see the final result
with open("seek_demo.txt", "r") as f:
    print(f"\nFinal file content: '{f.read()}'")

### Common Mistake & Key Takeaway

**Common Mistake**: After reading from a file in `r+` or `w+` mode, forgetting to `seek()` back to the beginning (or another desired position) before trying to read again. The pointer will be at the end of the file, and a subsequent `read()` will return an empty string.

**Key Takeaway**: `tell()` shows you where you are, and `seek()` lets you go somewhere else. You **must** manage the pointer manually when you need to read and write in the same session, especially in modes like `r+` and `w+`.

# Complete File Handling Cheat Sheet

### Core Principle
Always use the `with` statement for automatic and safe file closing.
```python
with open('filename.txt', 'r') as f:
    # do stuff with f
```

### File Modes

| Mode | Description | Reads? | Writes? | Creates? | Truncates? | Pointer Start |
| :--- | :--- | :---: | :---: | :---: | :---: | :--- |
| `r` | Read Only | ✅ | ❌ | ❌ | ❌ | Beginning |
| `w` | Write Only | ❌ | ✅ | ✅ | ✅ | Beginning |
| `a` | Append Only | ❌ | ✅ | ✅ | ❌ | End |
| `r+` | Read & Write | ✅ | ✅ | ❌ | ❌ | Beginning |
| `w+` | Write & Read | ✅ | ✅ | ✅ | ✅ | Beginning |
| `a+` | Append & Read | ✅ | ✅ | ✅ | ❌ | End |
| `rb` | Read Binary | ✅ | ❌ | ❌ | ❌ | Beginning |
| `wb` | Write Binary | ❌ | ✅ | ✅ | ✅ | Beginning |

### Reading Methods

| Method | Memory Usage | Best For |
| :--- | :--- | :--- |
| `for line in file:` | **Low** | The default, best choice for any file, especially large ones. |
| `file.read()` | **High** | Small files, get all content as one string. |
| `file.readlines()` | **High** | Small files, get all content as a list of lines. |
| `file.readline()` | **Low** | Processing a file one line at a time in a `while` loop. |

### Writing Methods

| Method | Input | Notes |
| :--- | :--- | :--- |
| `file.write(string)` | A single string | Does NOT add `\n` automatically. |
| `file.writelines(iterable)` | A list/iterable of strings | Does NOT add `\n` between elements. |

### Pointer/Cursor Control

| Method | Action |
| :--- | :--- |
| `file.tell()` | Returns current byte position (integer). |
| `file.seek(offset, from_what)` | Moves the pointer. `from_what`: 0=start, 1=current, 2=end. |